<a href="https://colab.research.google.com/github/sarah2005-cyber/auspex/blob/main/Preprocessing_steps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get install ffmpeg

!ffmpeg -codecs | grep 729



**CONVERTING .G729A AND PCM to .WAV**

**Directly to GDrive**

In [ ]:
import os
import subprocess

# Paths
input_cover = "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/Selected_Subset/cover"
input_stego = "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/Selected_Subset/stego"

output_base = "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/Selected_Subset/decoded_wav"

output_cover = os.path.join(output_base, "cover")
output_stego = os.path.join(output_base, "stego")

os.makedirs(output_cover, exist_ok=True)
os.makedirs(output_stego, exist_ok=True)

def decode_pcm(inp, out):
    subprocess.run([
        "ffmpeg", "-y",
        "-f", "s16le",
        "-ar", "8000",
        "-ac", "1",
        "-i", inp,
        "-ar", "44100",
        out
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

def decode_g729(inp, out):
    subprocess.run([
        "ffmpeg", "-y",
        "-f", "g729",
        "-i", inp,
        "-ar", "44100",
        "-ac", "1",
        out
    ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


def decode_folder(src, dst):
    for f in os.listdir(src):
        ip = os.path.join(src, f)
        if not os.path.isfile(ip):
            continue

        op = os.path.join(dst, f.rsplit(".", 1)[0] + ".wav")

        # Skip if already exists
        if os.path.exists(op):
            continue

        if f.endswith(".pcm"):
            decode_pcm(ip, op)
        elif f.endswith(".g729a"):
            decode_g729(ip, op)


In [ ]:
# Decode the cover folder
decode_folder(input_cover, output_cover)

# Decode the stego folder
decode_folder(input_stego, output_stego)


**Standardize audio**

In [ ]:
import soundfile as sf
import os
import numpy as np

TARGET_SR = 44100
TARGET_LEN = TARGET_SR * 1

def standardize_audio(input_path, output_dir, bit_depth="PCM_16", normalize=True):
    os.makedirs(output_dir, exist_ok=True)

    audio, sr = sf.read(input_path)

    # ---- Sanity check ----
    if sr != TARGET_SR:
        raise ValueError(f"{input_path} has sr={sr}, expected {TARGET_SR}")

    # ---- Ensure mono ----
    if audio.ndim > 1:
        audio = audio.mean(axis=1)

    # ---- Enforce fixed length (1s) ----
    if len(audio) > TARGET_LEN:
        audio = audio[:TARGET_LEN]
    elif len(audio) < TARGET_LEN:
        pad = TARGET_LEN - len(audio)
        audio = np.pad(audio, (0, pad), mode="constant")

    # ---- Optional peak normalization ----
    if normalize:
        peak = np.max(np.abs(audio))
        if peak > 0:
            audio = audio / peak

    # ---- Write standardized file ----
    out_path = os.path.join(
        output_dir,
        os.path.basename(input_path).replace(".wav", "_standardized.wav")
    )

    sf.write(out_path, audio, TARGET_SR, subtype=bit_depth)
    return out_path


In [ ]:
input_dir = "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/Selected_Subset/decoded_wav/stego"
std_dir = "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/Selected_Subset/standardized/stego"

for fname in os.listdir(input_dir):
    if fname.endswith(".wav"):
        out_file = os.path.join(std_dir, fname.replace(".wav", "_standardized.wav"))
        if os.path.exists(out_file):
            continue  # Skip already processed files
        standardize_audio(os.path.join(input_dir, fname), std_dir)


**MULTI WINDOW SPECTOGRAM STREAM PROCESSING**

**Check for matching files in stego and cover**

In [ ]:
# Filenames must be identical
!comm -3 \
  <(ls /content/local_standardized/cover | sort) \
  <(ls /content/local_standardized/stego | sort)

In [ ]:
import os
import librosa
import numpy as np

# ====== Parameters ======
cover_dir = "/content/local_standardized/stego"
# stego_dir = "/content/drive/MyDrive/.../stego_standardized"
all_dirs = [cover_dir]

sr = 44100
n_fft_1024, hop_1024 = 1024, 512
n_fft_512, hop_512 = 512, 256

max_frames_1024 = 0
max_frames_512 = 0

# ====== Scan all files ======
for d in all_dirs:
    for fname in os.listdir(d):
        if not fname.endswith(".wav"):
            continue

        file_path = os.path.join(d, fname)
        audio, _ = librosa.load(file_path, sr=sr, mono=True)

        # STFT 1024
        spec1024 = librosa.stft(audio, n_fft=n_fft_1024, hop_length=hop_1024, window="hamming")
        frames1024 = spec1024.shape[1]
        if frames1024 > max_frames_1024:
            max_frames_1024 = frames1024

        # STFT 512
        spec512 = librosa.stft(audio, n_fft=n_fft_512, hop_length=hop_512, window="hamming")
        frames512 = spec512.shape[1]
        if frames512 > max_frames_512:
            max_frames_512 = frames512

print("Max frames (1024-window) =", max_frames_1024)
print("Max frames (512-window)  =", max_frames_512)


Max frames (1024-window) = 87
Max frames (512-window)  = 173


**Fixed parameter filtering**

In [ ]:
import os
import numpy as np
import librosa
from scipy.signal import convolve2d

# =========================
# Paths (EDIT THESE)
# =========================
INPUT_DIR = "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/Selected_Subset/standardized/cover"

OUT_SPM1024  = "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/Selected_Subset/spec_1024_spm/cover"
OUT_SPM512   = "/content/drive/MyDrive/IIT - University of Westminster Material/VStego800K dataset/Selected_Subset/spec_512_spm/cover"

os.makedirs(OUT_SPM1024,  exist_ok=True)
os.makedirs(OUT_SPM512,   exist_ok=True)

# =========================
# Uniform time-frame targets
# =========================
MAX_FRAMES_1024 = 87
MAX_FRAMES_512  = 173

EPS = 1e-8

# =========================
# 1.3.1 + 1.3.2: Spectrogram (STFT + log + minmax + pad)
# =========================
def compute_log_norm_padded_spec(audio, sr, n_fft, hop_length, max_frames):

    # STFT
    S = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length, window="hamming")
    S_mag = np.abs(S)

    # Log magnitude
    S_log = np.log10(S_mag + EPS)

    # Min-max normalize per spectrogram
    s_min, s_max = S_log.min(), S_log.max()
    S_norm = (S_log - s_min) / (s_max - s_min + EPS)

    # Pad/trim along time axis
    freq_bins, time_frames = S_norm.shape
    if time_frames < max_frames:
        pad_t = max_frames - time_frames
        S_fixed = np.pad(S_norm, ((0, 0), (0, pad_t)), mode="constant")
    else:
        S_fixed = S_norm[:, :max_frames]

    return S_fixed

# =========================
# 1.3.3: Fixed-Parameter Spectrogram Filtering (SPM)
# =========================
KERNEL_STRONG = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1]
], dtype=np.float32)

KERNEL_LIGHT = np.array([
    [ 0, -1,  0],
    [-1,  4, -1],
    [ 0, -1,  0]
], dtype=np.float32)

def apply_spm_filters(spec, kernels, weights=None):
    if weights is None:
        weights = [1.0] * len(kernels)

    filtered_sum = np.zeros_like(spec)
    for ker, w in zip(kernels, weights):
        conv = convolve2d(spec, ker, mode="same", boundary="symm")
        filtered_sum += w * conv


    filtered_sum = np.maximum(filtered_sum, 0.0)


    fmin, fmax = filtered_sum.min(), filtered_sum.max()
    if fmax - fmin < EPS:
        return np.zeros_like(filtered_sum)
    return (filtered_sum - fmin) / (fmax - fmin + EPS)

# =========================
# Main loop
# =========================
for fname in os.listdir(INPUT_DIR):
    if not fname.lower().endswith(".wav"):
        continue

    path = os.path.join(INPUT_DIR, fname)
    audio, sr = librosa.load(path, sr=44100, mono=True)

    base = os.path.splitext(fname)[0]

    # ----- Window 1024 -----
    spec1024 = compute_log_norm_padded_spec(
        audio=audio, sr=sr, n_fft=1024, hop_length=512, max_frames=MAX_FRAMES_1024
    )


    spm1024 = apply_spm_filters(
        spec1024,
        kernels=[KERNEL_STRONG, KERNEL_LIGHT],
        weights=[1.0, 0.5]
    )
    np.save(os.path.join(OUT_SPM1024, f"{base}_spec1024_spm.npy"), spm1024)

    # ----- Window 512 -----
    spec512 = compute_log_norm_padded_spec(
        audio=audio, sr=sr, n_fft=512, hop_length=256, max_frames=MAX_FRAMES_512
    )

    spm512 = apply_spm_filters(
        spec512,
        kernels=[KERNEL_LIGHT],
        weights=[1.0]
    )
    np.save(os.path.join(OUT_SPM512, f"{base}_spec512_spm.npy"), spm512)

    print(
        f"{fname} -> "
        f"1024: raw {spec1024.shape}, spm {spm1024.shape} | "
        f"512: raw {spec512.shape}, spm {spm512.shape}"
    )
